In [ ]:
from mbientlab.metawear import MetaWear, libmetawear, parse_value
from mbientlab.metawear.cbindings import GyroBoschOdr, GyroBoschRange, FnVoid_VoidP_DataP
from mbientlab.metawear.cbindings import *
import time
import joblib
import numpy as np
from collections import deque
from scipy.signal import butter, lfilter


# --- 1. Load Pre-trained ML Pipeline ---
print("Loading ML models...")
scaler = joblib.load('scaler.pkl')
pca = joblib.load('pca.pkl')
model = joblib.load('best_model.pkl')

# --- 2. Load Config Parameters ---
# Values taken directly from your config.yml
WINDOW_SIZE = 120    # 100Hz = 1.2 seconds of data
OVERLAP = 60         # Slide by 60 samples
FILTER_ORDER = 4
FILTER_CUTOFF = 10   # Hz
SAMPLING_RATE = 100  # Hz

# --- 3. Setup Signal Processing ---
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = lfilter(b, a, data, axis=0)
    return y

# Thread-safe buffer for incoming data (Acc X, Y, Z + Gyro X, Y, Z)
data_buffer = deque(maxlen=WINDOW_SIZE)
current_sample = []

def process_and_predict(window_data):
    """Applies your exact ML pipeline to a single live window"""
    # 1. Convert to numpy array (Shape: 120, 6)
    raw_array = np.array(window_data)
    
    # 2. Apply Lowpass Filter
    filtered_array = butter_lowpass_filter(raw_array, FILTER_CUTOFF, SAMPLING_RATE, FILTER_ORDER)
    
    # 3. Flatten the window (Shape: 1, 720) - assuming 6 features * 120 samples
    flattened_window = filtered_array.flatten().reshape(1, -1)
    
    # 4. Scale
    scaled_window = scaler.transform(flattened_window)
    
    # 5. PCA Reduction
    pca_window = pca.transform(scaled_window)
    
    # 6. Predict!
    prediction = model.predict(pca_window)[0]
    print(f"\n>> LIVE PREDICTION: {prediction.upper()} <<\n")

# --- 4. MetaWear Callbacks ---
# We use a simple counter to pair Accel and Gyro readings coming in at 100Hz
sample_sync = {'acc': None, 'gyro': None}

def acc_callback(ctx, data):
    val = parse_value(data)
    sample_sync['acc'] = [val.x, val.y, val.z]
    check_sync()

def gyro_callback(ctx, data):
    val = parse_value(data)
    sample_sync['gyro'] = [val.x, val.y, val.z]
    check_sync()

def check_sync():
    # If we have both readings for this timestamp, append to buffer
    if sample_sync['acc'] is not None and sample_sync['gyro'] is not None:
        combined_features = sample_sync['acc'] + sample_sync['gyro']
        data_buffer.append(combined_features)
        
        # Reset for next sample
        sample_sync['acc'] = None
        sample_sync['gyro'] = None
        
        # Check if window is full and ready for prediction
        if len(data_buffer) == WINDOW_SIZE:
            # Copy buffer to process safely
            window_copy = list(data_buffer)
            process_and_predict(window_copy)
            
            # Pop the oldest samples to create the overlap (Overlap = 60)
            for _ in range(WINDOW_SIZE - OVERLAP):
                data_buffer.popleft()

# --- 5. Connect and Run ---
# --- 5. Connect and Run ---
MAC_ADDRESS = "E6:52:74:14:58:E5" # <-- INSERT YOUR MAC ADDRESS HERE

print(f"Initializing device {MAC_ADDRESS}...")
device = MetaWear(MAC_ADDRESS)

# --- ROBUST CONNECTION LOOP ---
max_retries = 5
connected = False

for attempt in range(max_retries):
    try:
        print(f"Connecting... (Attempt {attempt + 1}/{max_retries})")
        device.connect()
        connected = True
        print("Connected successfully! Configuring sensors to 100Hz...")
        break  # Exit the loop if connection succeeds!
    except Exception as e:
        print(f"Connection failed: {e}")
        if attempt < max_retries - 1:
            print("Retrying in 3 seconds...")
            time.sleep(3.0)
            
if not connected:
    print("\nFAILED TO CONNECT after 5 attempts.")
    print("Please turn Windows Bluetooth OFF and ON again, then restart script.")
    exit() # Stop the script

# Initialize variables safely to prevent NameError
acc_signal = None
gyro_signal = None
try:
    # Setup Accelerometer (100Hz, +/- 4g)
    # FIX: Use generic 'acc' functions, NOT 'acc_bmi160'
    libmetawear.mbl_mw_acc_set_odr(device.board, 100.0)
    libmetawear.mbl_mw_acc_set_range(device.board, 4.0)
    libmetawear.mbl_mw_acc_write_acceleration_config(device.board)
    
    # Setup Gyroscope (100Hz, 2000 degrees/s)
    libmetawear.mbl_mw_gyro_bmi160_set_odr(device.board, GyroBmi160Odr._100Hz)
    libmetawear.mbl_mw_gyro_bmi160_set_range(device.board, GyroBmi160Range._2000dps)
    libmetawear.mbl_mw_gyro_write_config(device.board)

    # Subscribe to data streams
    acc_signal = libmetawear.mbl_mw_acc_get_acceleration_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(acc_signal, None, FnVoid_VoidP_DataP(acc_callback))
    
    gyro_signal = libmetawear.mbl_mw_gyro_bmi160_get_rotation_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(gyro_signal, None, FnVoid_VoidP_DataP(gyro_callback))

    # Start sensors
    print("Starting data collection. Perform gestures now! (Press Ctrl+C to stop)")
    libmetawear.mbl_mw_acc_enable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_acc_start(device.board)
    libmetawear.mbl_mw_gyro_bmi160_enable_rotation_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_start(device.board)

    # Keep script running
    while True:
        time.sleep(1.0)

except KeyboardInterrupt:
    print("\nStopping...")
finally:
    # Safely teardown connection
    libmetawear.mbl_mw_acc_stop(device.board)
    libmetawear.mbl_mw_acc_disable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_stop(device.board)
    libmetawear.mbl_mw_gyro_bmi160_disable_rotation_sampling(device.board)
    
    # FIX: Only unsubscribe if they were successfully created
    if acc_signal is not None:
        libmetawear.mbl_mw_datasignal_unsubscribe(acc_signal)
    if gyro_signal is not None:
        libmetawear.mbl_mw_datasignal_unsubscribe(gyro_signal)
    
    device.disconnect()
    print("Disconnected safely.")

Loading ML models...
Initializing device E6:52:74:14:58:E5...
Connecting... (Attempt 1/5)
Connection failed: Failed to discover gatt services (status = 1)
Retrying in 3 seconds...
Connecting... (Attempt 2/5)


In [ ]:
from mbientlab.metawear import MetaWear, libmetawear, parse_value
from mbientlab.metawear.cbindings import GyroBoschOdr, GyroBoschRange, FnVoid_VoidP_DataP
import time
import joblib
import numpy as np
from collections import deque
from scipy.signal import butter, lfilter

# --- 1. Load Pre-trained ML Pipeline ---
print("Loading ML models...")
scaler = joblib.load('scaler.pkl')
pca = joblib.load('pca.pkl')
model = joblib.load('best_model.pkl')

# --- 2. Load Config Parameters ---
WINDOW_SIZE = 120    # 100Hz = 1.2 seconds of data
OVERLAP = 60         # Slide by 60 samples
FILTER_ORDER = 4
FILTER_CUTOFF = 10   # Hz
SAMPLING_RATE = 100  # Hz

# --- 3. Setup Signal Processing ---
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = lfilter(b, a, data, axis=0)
    return y

data_buffer = deque(maxlen=WINDOW_SIZE)
sample_sync = {'acc': None, 'gyro': None}

def process_and_predict(window_data):
    raw_array = np.array(window_data)
    filtered_array = butter_lowpass_filter(raw_array, FILTER_CUTOFF, SAMPLING_RATE, FILTER_ORDER)
    flattened_window = filtered_array.flatten().reshape(1, -1)
    scaled_window = scaler.transform(flattened_window)
    pca_window = pca.transform(scaled_window)
    prediction = model.predict(pca_window)[0]
    print(f"\n>> LIVE PREDICTION: {prediction.upper()} <<\n")

# --- 4. MetaWear Callbacks ---
def acc_callback(ctx, data):
    val = parse_value(data)
    sample_sync['acc'] = [val.x, val.y, val.z]
    check_sync()

def gyro_callback(ctx, data):
    val = parse_value(data)
    sample_sync['gyro'] = [val.x, val.y, val.z]
    check_sync()

def check_sync():
    if sample_sync['acc'] is not None and sample_sync['gyro'] is not None:
        combined_features = sample_sync['acc'] + sample_sync['gyro']
        data_buffer.append(combined_features)
        
        sample_sync['acc'] = None
        sample_sync['gyro'] = None
        
        if len(data_buffer) == WINDOW_SIZE:
            window_copy = list(data_buffer)
            process_and_predict(window_copy)
            for _ in range(WINDOW_SIZE - OVERLAP):
                data_buffer.popleft()

# --- 5. Connect and Run ---
MAC_ADDRESS = "E6:52:74:14:58:E5"

print(f"Initializing device {MAC_ADDRESS}...")
device = MetaWear(MAC_ADDRESS)

max_retries = 5
connected = False

for attempt in range(max_retries):
    try:
        print(f"Connecting... (Attempt {attempt + 1}/{max_retries})")
        device.connect()
        connected = True
        print("Connected successfully! Configuring sensors to 100Hz...")
        break
    except Exception as e:
        print(f"Connection failed: {e}")
        if attempt < max_retries - 1:
            print("Retrying in 3 seconds...")
            time.sleep(3.0)
            
if not connected:
    print("\nFAILED TO CONNECT after 5 attempts.")
    exit()

acc_signal = None
gyro_signal = None

try:
    # Setup Accelerometer (100Hz, +/- 4g)
    libmetawear.mbl_mw_acc_set_odr(device.board, 100.0)
    libmetawear.mbl_mw_acc_set_range(device.board, 4.0)
    libmetawear.mbl_mw_acc_write_acceleration_config(device.board)
    
    # Setup Gyroscope (100Hz, 2000 degrees/s) using the new Bosch variables
    libmetawear.mbl_mw_gyro_bmi160_set_odr(device.board, GyroBoschOdr._100Hz)
    libmetawear.mbl_mw_gyro_bmi160_set_range(device.board, GyroBoschRange._2000dps)
    libmetawear.mbl_mw_gyro_write_config(device.board)

    # Subscribe to data streams
    acc_signal = libmetawear.mbl_mw_acc_get_acceleration_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(acc_signal, None, FnVoid_VoidP_DataP(acc_callback))
    
    gyro_signal = libmetawear.mbl_mw_gyro_bmi160_get_rotation_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(gyro_signal, None, FnVoid_VoidP_DataP(gyro_callback))

    print("Starting data collection. Perform gestures now! (Press Ctrl+C to stop)")
    libmetawear.mbl_mw_acc_enable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_acc_start(device.board)
    libmetawear.mbl_mw_gyro_bmi160_enable_rotation_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_start(device.board)

    while True:
        time.sleep(1.0)

except KeyboardInterrupt:
    print("\nStopping...")
finally:
    libmetawear.mbl_mw_acc_stop(device.board)
    libmetawear.mbl_mw_acc_disable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_stop(device.board)
    libmetawear.mbl_mw_gyro_bmi160_disable_rotation_sampling(device.board)
    
    if acc_signal is not None:
        libmetawear.mbl_mw_datasignal_unsubscribe(acc_signal)
    if gyro_signal is not None:
        libmetawear.mbl_mw_datasignal_unsubscribe(gyro_signal)
    
    device.disconnect()
    print("Disconnected safely.")

Loading ML models...
Initializing device E6:52:74:14:58:E5...
Connecting... (Attempt 1/5)


In [1]:
"""
live_gesture_test.py
====================
Real-time gesture recognition using a trained HGR pipeline.

Supports two modes:
  --mode sensor   : Connect to MetaMotionR via BLE (requires metawear SDK)
  --mode simulate : Generate synthetic IMU data to test the pipeline offline

Usage
-----
  python live_gesture_test.py --mode simulate
  python live_gesture_test.py --mode sensor --address AA:BB:CC:DD:EE:FF

Requirements
------------
  pip install numpy scipy scikit-learn joblib pyyaml
  pip install metawear          # only needed for --mode sensor
"""

import argparse
import time
import os
import sys
import collections

import numpy as np
import joblib
import yaml
from scipy.signal import butter, filtfilt

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

WINDOW_SIZE   = 120          # must match training ws in config.yml
SENSOR_COLS   = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
SAMPLING_RATE = 100          # Hz — adjust to match your config.yml sr value

# Paths to saved pipeline artefacts (same folder as this script by default)
_HERE = os.path.dirname(os.path.abspath(__file__))
SCALER_PATH = os.path.join(_HERE, "scaler.pkl")
PCA_PATH    = os.path.join(_HERE, "pca.pkl")
MODEL_PATH  = os.path.join(_HERE, "best_model.pkl")
CONFIG_PATH = os.path.join(_HERE, "config.yml")

# ─────────────────────────────────────────────
# LOAD PIPELINE
# ─────────────────────────────────────────────

def load_pipeline():
    """Load scaler, PCA and classifier saved during training."""
    print("[INIT] Loading saved pipeline …")
    for path, label in [(SCALER_PATH, "scaler"), (PCA_PATH, "PCA"), (MODEL_PATH, "model")]:
        if not os.path.exists(path):
            sys.exit(f"[ERROR] Cannot find {label} at: {path}\n"
                     "        Make sure scaler.pkl, pca.pkl and best_model.pkl "
                     "are in the same folder as this script.")
    scaler = joblib.load(SCALER_PATH)
    pca    = joblib.load(PCA_PATH)
    model  = joblib.load(MODEL_PATH)
    print(f"[INIT] Pipeline loaded — classes: {list(model.classes_)}")
    return scaler, pca, model


def load_filter_config():
    """Read filter parameters from config.yml (mirrors the notebook)."""
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH) as f:
            cfg = yaml.safe_load(f)
        order   = cfg["filter"]["order"]
        cutoff  = cfg["filter"]["wn"]
        ftype   = cfg["filter"]["type"]
        sr      = cfg.get("sr", SAMPLING_RATE)
        print(f"[INIT] Filter config — order={order}, cutoff={cutoff} Hz, type={ftype}, sr={sr}")
        return order, cutoff, ftype, sr
    else:
        # Sensible defaults matching the project description
        print("[INIT] config.yml not found — using defaults (order=4, cutoff=10 Hz, lowpass, sr=100)")
        return 4, 10.0, "lowpass", SAMPLING_RATE

# ─────────────────────────────────────────────
# SIGNAL PROCESSING  (mirrors notebook exactly)
# ─────────────────────────────────────────────

def apply_lowpass(arr: np.ndarray, order: int, cutoff: float, sr: int) -> np.ndarray:
    """Butterworth low-pass filter — identical logic to the notebook."""
    nyquist    = 0.5 * sr
    normalized = cutoff / nyquist
    normalized = np.clip(normalized, 1e-6, 1 - 1e-6)   # safety clamp
    b, a = butter(order, normalized, btype="low")
    return filtfilt(b, a, arr)


def preprocess_window(window: np.ndarray,
                      order: int, cutoff: float, sr: int,
                      scaler, pca) -> np.ndarray:
    """
    Full preprocessing chain applied to a single (WINDOW_SIZE × 6) array:
      1. Low-pass filter each channel
      2. Flatten → 1-D feature vector
      3. StandardScaler
      4. PCA
    Returns a 2-D array of shape (1, n_components) ready for model.predict().
    """
    filtered = np.zeros_like(window, dtype=float)
    for ch in range(window.shape[1]):
        filtered[:, ch] = apply_lowpass(window[:, ch], order, cutoff, sr)

    flat = filtered.flatten().reshape(1, -1)       # shape: (1, WINDOW_SIZE*6)
    scaled = scaler.transform(flat)
    reduced = pca.transform(scaled)
    return reduced


# ─────────────────────────────────────────────
# PREDICTION  (shared by both modes)
# ─────────────────────────────────────────────

def predict_gesture(window: np.ndarray,
                    order, cutoff, sr,
                    scaler, pca, model) -> tuple[str, float]:
    """Returns (predicted_label, confidence_percent)."""
    features = preprocess_window(window, order, cutoff, sr, scaler, pca)
    label    = model.predict(features)[0]

    # Confidence: use predict_proba if available, else decision function
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(features)[0]
        conf  = float(np.max(proba)) * 100
    elif hasattr(model, "decision_function"):
        dec  = model.decision_function(features)[0]
        conf = float(np.max(np.abs(dec)))   # raw margin — not a probability
    else:
        conf = 0.0

    return label, conf


# ─────────────────────────────────────────────
# SIMULATE MODE
# ─────────────────────────────────────────────

def simulate_imu_sample(t: float) -> list[float]:
    """
    Generate a realistic-looking 6-axis IMU sample.
    Each call returns [acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z].
    """
    acc_x = 0.3 * np.sin(2 * np.pi * 1.5 * t) + np.random.normal(0, 0.05)
    acc_y = 0.5 * np.cos(2 * np.pi * 1.0 * t) + np.random.normal(0, 0.05)
    acc_z = 9.81 + 0.2 * np.sin(2 * np.pi * 0.5 * t) + np.random.normal(0, 0.02)
    gyr_x = 15  * np.sin(2 * np.pi * 2.0 * t) + np.random.normal(0, 1.0)
    gyr_y = 10  * np.cos(2 * np.pi * 1.5 * t) + np.random.normal(0, 1.0)
    gyr_z = 5   * np.sin(2 * np.pi * 0.8 * t) + np.random.normal(0, 0.5)
    return [acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z]


def run_simulate(scaler, pca, model, order, cutoff, sr,
                 n_predictions: int = 20, delay: float = 0.05):
    """
    Simulate a live stream: fill a rolling buffer, predict every WINDOW_SIZE samples.
    """
    print("\n" + "="*60)
    print("  SIMULATE MODE — generating synthetic IMU data")
    print(f"  Window size : {WINDOW_SIZE} samples")
    print(f"  Predictions : {n_predictions}")
    print("="*60 + "\n")

    buffer = collections.deque(maxlen=WINDOW_SIZE)
    sample_idx = 0
    prediction_count = 0

    while prediction_count < n_predictions:
        t = sample_idx / sr
        sample = simulate_imu_sample(t)
        buffer.append(sample)
        sample_idx += 1

        if len(buffer) == WINDOW_SIZE:
            window = np.array(buffer)           # shape: (120, 6)
            label, conf = predict_gesture(window, order, cutoff, sr, scaler, pca, model)

            ts = time.strftime("%H:%M:%S")
            bar = "█" * int(conf / 5) if conf <= 100 else "█" * 20
            print(f"[{ts}]  Gesture: {label:<15}  Confidence: {conf:5.1f}%  {bar}")

            prediction_count += 1
            buffer.clear()       # non-overlapping windows for cleaner demo

        time.sleep(delay)

    print("\n[DONE] Simulation complete.")


# ─────────────────────────────────────────────
# SENSOR MODE  (MetaMotionR via BLE)
# ─────────────────────────────────────────────

def run_sensor(address: str, scaler, pca, model, order, cutoff, sr):
    """
    Stream live IMU data from MetaMotionR and predict gestures in real time.

    Requires:  pip install metawear
    Find MAC:  Use the MetaWear app or 'bluetoothctl scan on' on Linux.
    """
    try:
        from mbientlab.metawear import MetaWear, libmetawear, parse_value, cbindings
        from mbientlab.metawear.cbindings import (
            SensorFusionData, SensorFusionMode, SensorFusionAccRange,
            SensorFusionGyroRange, FnVoid_VoidP_DataP
        )
    except ImportError:
        sys.exit("[ERROR] metawear SDK not installed.\n"
                 "        Run: pip install metawear\n"
                 "        Or use --mode simulate to test without the sensor.")

    print("\n" + "="*60)
    print(f"  SENSOR MODE — connecting to {address}")
    print(f"  Window size : {WINDOW_SIZE} samples  |  SR: {sr} Hz")
    print("  Press Ctrl+C to stop.")
    print("="*60 + "\n")

    device = MetaWear(address)
    device.connect()
    print(f"[BLE]  Connected — firmware {device.info.firmware_revision}")

    buffer = collections.deque(maxlen=WINDOW_SIZE)
    sample_count = [0]

    def data_handler(ctx, data):
        value = parse_value(data)
        # MetaWear returns (acc_x, acc_y, acc_z) for accelerometer
        # Gyroscope must be subscribed separately — see below
        pass   # placeholder; real handler is defined per-subscription

    # ── Subscribe to Accelerometer ──────────────────────────────────────────
    acc_buf = collections.deque(maxlen=WINDOW_SIZE)
    gyr_buf = collections.deque(maxlen=WINDOW_SIZE)

    def on_acc(ctx, data):
        v = parse_value(data)
        acc_buf.append([v.x, v.y, v.z])

    def on_gyr(ctx, data):
        v = parse_value(data)
        gyr_buf.append([v.x, v.y, v.z])
        sample_count[0] += 1

        # Once we have a full window of both acc + gyr, predict
        if len(acc_buf) == WINDOW_SIZE and len(gyr_buf) == WINDOW_SIZE:
            window = np.hstack([np.array(acc_buf), np.array(gyr_buf)])  # (120, 6)
            label, conf = predict_gesture(window, order, cutoff, sr, scaler, pca, model)

            ts = time.strftime("%H:%M:%S")
            bar = "█" * int(conf / 5) if conf <= 100 else "█" * 20
            print(f"[{ts}]  Gesture: {label:<15}  Confidence: {conf:5.1f}%  {bar}")

            acc_buf.clear()
            gyr_buf.clear()

    # Convert Python callbacks to C-compatible function pointers
    fn_acc = FnVoid_VoidP_DataP(on_acc)
    fn_gyr = FnVoid_VoidP_DataP(on_gyr)

    # Configure and start accelerometer
    libmetawear.mbl_mw_acc_set_odr(device.board, float(sr))
    libmetawear.mbl_mw_acc_set_range(device.board, 4.0)      # ±4g
    libmetawear.mbl_mw_acc_write_acceleration_config(device.board)
    signal_acc = libmetawear.mbl_mw_acc_get_acceleration_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(signal_acc, None, fn_acc)
    libmetawear.mbl_mw_acc_enable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_acc_start(device.board)

    # Configure and start gyroscope (BMI160)
    libmetawear.mbl_mw_gyro_bmi160_set_odr(device.board, cbindings.GyroBoschOdr._100Hz)
    libmetawear.mbl_mw_gyro_bmi160_set_range(device.board, cbindings.GyroBoschRange._500dps)
    libmetawear.mbl_mw_gyro_bmi160_write_config(device.board)
    signal_gyr = libmetawear.mbl_mw_gyro_bmi160_get_rotation_data_signal(device.board)
    libmetawear.mbl_mw_datasignal_subscribe(signal_gyr, None, fn_gyr)
    libmetawear.mbl_mw_gyro_bmi160_enable_rotation_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_start(device.board)

    print("[STREAM] Streaming — perform a gesture now …\n")

    try:
        while True:
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("\n[STOP] Stopping sensor stream …")

    # Tear down
    libmetawear.mbl_mw_acc_stop(device.board)
    libmetawear.mbl_mw_acc_disable_acceleration_sampling(device.board)
    libmetawear.mbl_mw_gyro_bmi160_stop(device.board)
    libmetawear.mbl_mw_gyro_bmi160_disable_rotation_sampling(device.board)
    libmetawear.mbl_mw_datasignal_unsubscribe(signal_acc)
    libmetawear.mbl_mw_datasignal_unsubscribe(signal_gyr)
    device.disconnect()
    print("[BLE]  Disconnected.")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Live HGR gesture recognition")
    parser.add_argument("--mode",    choices=["sensor", "simulate"], default="simulate",
                        help="'sensor' streams from MetaMotionR BLE; 'simulate' uses synthetic data")
    parser.add_argument("--address", default=None,
                        help="BLE MAC address of MetaMotionR (required for --mode sensor)")
    parser.add_argument("--predictions", type=int, default=20,
                        help="Number of predictions to make in simulate mode (default: 20)")
    args = parser.parse_args()

    if args.mode == "sensor" and not args.address:
        sys.exit("[ERROR] --mode sensor requires --address AA:BB:CC:DD:EE:FF")

    scaler, pca, model = load_pipeline()
    order, cutoff, ftype, sr = load_filter_config()

    if args.mode == "simulate":
        run_simulate(scaler, pca, model, order, cutoff, sr,
                     n_predictions=args.predictions)
    else:
        run_sensor(args.address, scaler, pca, model, order, cutoff, sr)


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined